In [ ]:
%pip install -U langchain-groq langchain_community pypdf langchain_chroma
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, ToolMessage, SystemMessage, HumanMessage
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_community.document_loaders import PyPDFLoader
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

In [ ]:
%%writefile .env
GROQ_API_KEY=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
load_dotenv(".env", override=True)

Overwriting .env


In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.0,
    api_key=''
)

In [ ]:
%pip install -U langchain-huggingface sentence-transformers
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
from google.colab import files
uploaded = files.upload()

pdf_path="Stock_Market_Performance_2024.pdf"
if not os.path.exists(pdf_path):
  raise FileNotFoundError(f"PDF file not found: {pdf_path}")

Saving Stock_Market_Performance_2024.pdf to Stock_Market_Performance_2024 (5).pdf


In [ ]:
pdf_loader=PyPDFLoader(pdf_path)
try:
  pages=pdf_loader.load()
  print(f"PDF has been loaded and has {len(pages)} pages.")
except Exception as e:
  print(f"Error loading PDF: {e}")
  raise

PDF has been loaded and has 9 pages.


In [ ]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [ ]:
pages_split=text_splitter.split_documents(pages)
persist_directory=r"/Users/sushilbajpai/Documents/Agents"
collection_name="stock_market"

In [ ]:
if not os.path.exists(persist_directory):
  os.makedirs(persist_directory)

In [ ]:
try:
  vectorstore=Chroma.from_documents(
      documents=pages_split,
      embedding=embeddings,
      persist_directory=persist_directory,
      collection_name=collection_name
  )
  print(f"Created ChromaDB vector store!")

except Exception as e:
  print(f"Error setting up chroma db: {str(e)}")
  raise

Created ChromaDB vector store!


In [ ]:
retriever=vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [ ]:
@tool
def retriever_tool(query: str) -> str:
  """ This tool searches and returns the information from document """
  docs=retriever.invoke(query)
  if not docs:
    return "I found no relevant information in the provided document"
  results=[]
  for i, doc in enumerate(docs):
    results.append(f"Document {i+1}:\n{doc.page_content}")
  return "\n\n".join(results)

In [ ]:
tools=[retriever_tool]
llm=llm.bind_tools(tools)

In [ ]:
class AgentState(TypedDict):
  messages: Annotated[Sequence[BaseMessage], add_messages]

In [ ]:
def should_continue(state: AgentState):
  """Check if the last message contain tool calls"""
  result=state['messages'][-1]
  return hasattr(result, 'tool_calls') and len(result.tool_calls)>0

In [ ]:
system_prompt="""You are an intelligent AI assistant who answers questions about stock market performance. User retriever tools to answer stock market data. Please always use the specific parts of the documents you use in your answers."""

In [ ]:
tools_dict = {our_tool.name: our_tool for our_tool in tools}

In [ ]:
def call_llm(state: AgentState) -> AgentState:
  """Function to call LLM with current state"""
  messages=list(state['messages'])
  messages=[SystemMessage(content=system_prompt)]+messages
  message=llm.invoke(messages)
  return {"messages": [message]}

In [ ]:
def take_action(state: AgentState):
  """Execute tool calls with LLM response"""
  tool_calls=state['messages'][-1].tool_calls
  results=[]
  for t in tool_calls:
    print(f"Calling tool: {t['name']} with query: {t['args'].get('query', 'No query provided')}")
    if not t['name'] in tools_dict:
      print(f"\nTool: {t['name']} does not exist.")
      result="Incorrect tool name, Please retry and select tools from list of available tools"
    else:
      result=tools_dict[t['name']].invoke(t['args'].get('query', ''))
      print(f"Result length: {len(str(result))}")

    results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
  print("Tools execution complete. Back to the model.")
  return {"messages": results}

In [ ]:
graph=StateGraph(AgentState)
graph.add_node("llm", call_llm)
graph.add_node("retriever_agent", take_action)
graph.add_conditional_edges(
    "llm",
    should_continue,
    {True: "retriever_agent", False: END}
)
graph.add_edge("retriever_agent", "llm")
graph.set_entry_point("llm")

In [ ]:
rag_agent=graph.compile()

In [ ]:
def running_agent():
  print("\n RAG Agent")
  while True:
    user_input=input("\nWhat is your question: ")
    if user_input.lower() in ['exit', 'quit']:
      break
    messages=[HumanMessage(content=user_input)]
    result=rag_agent.invoke({"messages": messages})
    print("=========ANSWER======")
    print(result['messages'][-1].content)

running_agent()


 RAG Agent
Calling tool: retriever_tool with query: IonQ profitability
Result length: 4677
Tools execution complete. Back to the model.
=========ANSWER======
IonQ has **not yet turned a profit**.  According to recent coverage, the company’s revenue—still only in the tens of millions of dollars—nearly doubled in 2024, but like many early‑stage tech firms it remained **unprofitable**.  The market has largely focused on its long‑term potential rather than current losses【1†L1-L5】【2†L1-L5】.
